## Notebook Overview — Preprocess Images

### **Purpose**

This notebook preprocesses raw images for a **user-selected dataset source** into a standardized format for downstream feature extraction and classification. The notebook processes **one source at a time**, allowing controlled execution within the Colab environment.

---

### **User Input**

At runtime, the user selects a single dataset source to preprocess from the available options.

Available sources in this notebook:

* **REAL sources**

  * ImageNet
  * MS COCO

* **AI sources**

  * DiffusionDB
  * SDXL
  * Midjourney

**Note:** OpenImages is excluded from this notebook workflow because its raw ZIP file is too large for practical Colab-based local extraction.

---

### **Inputs**

* A user-selected raw source ZIP file stored on Google Drive
* Shared project constants imported from `project_config.py`

Typical ZIP source location:

* `releases/raw/<selected_source>.zip`

The raw images stored in the ZIP file already follow the project filename convention established during dataset collection. These images are **not yet preprocessed**, but they have already been saved using standardized project filenames.

---

### **Processing Flow**

* Accept user input for the dataset source
* Display ZIP file size for each available source option
* Locate the selected ZIP file on Google Drive
* Copy the ZIP file to the local Colab runtime with progress display
* Extract the ZIP locally into `/content/raw_extracted/`
* Detect the extracted image directory
* Resize images to **256 × 256**
* Convert images to **grayscale**
* Standardize image output format
* Save preprocessed images using the **same filenames** as the raw stored images
* Save metadata for the selected source

---

### **Filename Behavior**

The raw images extracted from the ZIP file already use the predefined project naming convention, for example:

* `rl_imgn_000001.png`
* `ai_diff_000001.png`

Because these raw stored filenames are already standardized, the preprocessing step preserves the same filename when writing the processed image to the preprocessed output folder. The filename remains the same, while the image content is transformed through resizing and grayscale conversion.

---

### **Outputs**

* Preprocessed images written to a temporary local runtime directory:

  * `/content/preprocessed/<dataset>/images/`

  These files exist only for the duration of the Colab session and are not persisted in the GitHub repository.

* Preprocessed metadata CSV written to:

  * `metadata/preprocessed/<dataset>_preprocessed_metadata.csv`

Each metadata row records:

* Filename
* Class label
* Dataset code
* Source identifiers
* Original image dimensions
* Processed image path

---

### **Design Principles**

* **User-driven execution** — preprocess only the selected source
* **ZIP-based input handling** — source files are retrieved from Google Drive archives
* **Local runtime extraction** — ZIP is copied and extracted in `/content`
* **Source isolation** — no dataset mixing occurs in this notebook
* **Filename preservation** — raw stored filename and preprocessed filename remain the same
* **Metadata-driven workflow** — downstream stages rely on CSV metadata
* **Config-driven design** — shared constants and paths come from `project_config.py`
* **Fresh-run assumption** — notebook is intended to run cleanly for one selected source at a time

---

### **Assumptions**

* Raw source ZIP files have already been created and stored on Google Drive
* Raw stored images already follow the project filename convention
* The notebook is run in Google Colab with Google Drive mounted
* Sufficient local runtime storage is available for ZIP copy and extraction

---

### **Next Step**

The per-source metadata outputs from this notebook are later combined across all sources in:

➡️ `03_Combine_and_Split.ipynb`

That notebook merges the source-level metadata files and creates the final training and testing metadata splits.

---
---

### **Step 1 — Environment Setup and Configuration**

* Mounts Google Drive
* Imports required libraries
* Adds the project `src` directory to the Python path
* Imports shared constants from `project_config.py`

In [ ]:
# ============================================
# Step 1: Environment Setup and Configuration
# ============================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import zipfile
import shutil
import importlib.util

import pandas as pd
from PIL import Image

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # User toggle (True or False)

# -------------------------------------------------
# Define project locations
# -------------------------------------------------
BASE_DIR = "/content/dip-ai-image-detection"
PROJECT_SRC_DIR = f"{BASE_DIR}/src"
CONFIG_FILE = f"{PROJECT_SRC_DIR}/project_config.py"

print("Initializing environment...")

# -------------------------------------------------
# Clone repo if not already present
# -------------------------------------------------
if not os.path.isdir(BASE_DIR):
    print("Cloning repository...")
    !git clone -q https://github.com/pgailinas/dip-ai-image-detection.git
else:
    print("Repository already exists. Skipping clone.")

# -------------------------------------------------
# Verify structure
# -------------------------------------------------
if not os.path.isdir(PROJECT_SRC_DIR):
    raise FileNotFoundError(f"Missing directory: {PROJECT_SRC_DIR}")

if not os.path.isfile(CONFIG_FILE):
    raise FileNotFoundError(f"Missing file: {CONFIG_FILE}")

# -------------------------------------------------
# Add src to path
# -------------------------------------------------
if PROJECT_SRC_DIR not in sys.path:
    sys.path.insert(0, PROJECT_SRC_DIR)

# -------------------------------------------------
# Import config
# -------------------------------------------------
spec = importlib.util.spec_from_file_location("project_config", CONFIG_FILE)
cfg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cfg)

# -------------------------------------------------
# Pull only what this notebook actually uses
# -------------------------------------------------
LOCAL_WORK_DIR = cfg.LOCAL_WORK_DIR

RAW_RELEASES_DIR = cfg.RAW_RELEASES_DIR
RAW_ZIP_PATHS = cfg.RAW_ZIP_PATHS

DATASET_CODES = cfg.DATASET_CODES
SOURCE_FOLDER_NAMES = cfg.SOURCE_FOLDER_NAMES
SOURCE_LABEL_MAP = cfg.SOURCE_LABEL_MAP

PROCESSED_DATA_DIR = cfg.PROCESSED_DATA_DIR
PREPROCESSED_DIRS = cfg.PREPROCESSED_DIRS
PREPROCESSED_METADATA_DIR = cfg.PREPROCESSED_METADATA_DIR
PREPROCESSED_METADATA_FILES = cfg.PREPROCESSED_METADATA_FILES
PREPROCESSED_METADATA_COLUMNS = cfg.PREPROCESSED_METADATA_COLUMNS

TARGET_SIZE = cfg.TARGET_SIZE
OUTPUT_IMAGE_FORMAT = cfg.OUTPUT_IMAGE_FORMAT
OUTPUT_COLOR_MODE = cfg.OUTPUT_COLOR_MODE
OUTPUT_FILE_EXTENSION = cfg.OUTPUT_FILE_EXTENSION

# -------------------------------------------------
# Ensure required directories exist
# -------------------------------------------------
os.makedirs(cfg.DATA_DIR, exist_ok=True)
os.makedirs(cfg.PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(cfg.METADATA_DIR, exist_ok=True)
os.makedirs(cfg.PREPROCESSED_METADATA_DIR, exist_ok=True)

print("\nEnvironment setup complete.\n")

if VERBOSE:
    print(f"BASE_DIR                  : {cfg.BASE_DIR}")
    print(f"PROJECT_SRC_DIR           : {PROJECT_SRC_DIR}")
    print(f"RAW_RELEASES_DIR          : {RAW_RELEASES_DIR}")
    print(f"PROCESSED_DATA_DIR        : {PROCESSED_DATA_DIR}")
    print(f"PREPROCESSED_METADATA_DIR : {PREPROCESSED_METADATA_DIR}")



### **Step 2 — User Source Selection**

* Displays available dataset source options
* Shows ZIP file sizes for each selectable source
* Accepts user input for which source to preprocess
* Maps the selected source to:

  * class label
  * dataset code
  * ZIP filename
  * dataset folder

In [ ]:
# ============================================
# Step 2: User Source Selection (with ZIP sizes)
# ============================================

# -------------------------------------------------
# Base path for ZIP files (Google Drive)
# -------------------------------------------------
ZIP_BASE_DIR = RAW_RELEASES_DIR

# -------------------------------------------------
# Helper: Get file size in GB
# -------------------------------------------------
def get_zip_size_gb(filepath):
    if not os.path.exists(filepath):
        return None
    size_bytes = os.path.getsize(filepath)
    return size_bytes / (1024 ** 3)

# -------------------------------------------------
# Available dataset source options
# (OpenImages excluded due to size)
# -------------------------------------------------
SOURCE_OPTIONS = {
    "1": {
        "source_name": "imagenet",
        "display_name": "ImageNet",
        "dataset_folder": SOURCE_FOLDER_NAMES["imagenet"],
        "zip_filename": os.path.basename(RAW_ZIP_PATHS["imagenet"]),
        "dataset_code": DATASET_CODES["imagenet"],
        "class_label": SOURCE_LABEL_MAP["imagenet"],
    },
    "2": {
        "source_name": "coco",
        "display_name": "MS COCO",
        "dataset_folder": SOURCE_FOLDER_NAMES["coco"],
        "zip_filename": os.path.basename(RAW_ZIP_PATHS["coco"]),
        "dataset_code": DATASET_CODES["coco"],
        "class_label": SOURCE_LABEL_MAP["coco"],
    },
    "3": {
        "source_name": "diffusiondb",
        "display_name": "DiffusionDB",
        "dataset_folder": SOURCE_FOLDER_NAMES["diffusiondb"],
        "zip_filename": os.path.basename(RAW_ZIP_PATHS["diffusiondb"]),
        "dataset_code": DATASET_CODES["diffusiondb"],
        "class_label": SOURCE_LABEL_MAP["diffusiondb"],
    },
    "4": {
        "source_name": "sdxl",
        "display_name": "SDXL",
        "dataset_folder": SOURCE_FOLDER_NAMES["sdxl"],
        "zip_filename": os.path.basename(RAW_ZIP_PATHS["sdxl"]),
        "dataset_code": DATASET_CODES["sdxl"],
        "class_label": SOURCE_LABEL_MAP["sdxl"],
    },
    "5": {
        "source_name": "midjourney",
        "display_name": "Midjourney",
        "dataset_folder": SOURCE_FOLDER_NAMES["midjourney"],
        "zip_filename": os.path.basename(RAW_ZIP_PATHS["midjourney"]),
        "dataset_code": DATASET_CODES["midjourney"],
        "class_label": SOURCE_LABEL_MAP["midjourney"],
    },
}

# -------------------------------------------------
# Display choices with ZIP sizes
# -------------------------------------------------
print("Select one dataset source to preprocess:\n")

for key, info in SOURCE_OPTIONS.items():
    source_name = info["source_name"]
    zip_path = RAW_ZIP_PATHS[source_name]
    size_gb = get_zip_size_gb(zip_path)

    if size_gb is None:
        size_str = "NOT FOUND"
    else:
        size_str = f"{size_gb:.2f} GB"

    warning = " ⚠️ LARGE" if size_gb and size_gb > 5 else ""

    print(f"{key} - {info['display_name']} ({size_str}){warning}")

print("\nNote: OpenImages (~22GB) is excluded due to size constraints.")

# -------------------------------------------------
# User selection
# -------------------------------------------------
user_choice = input("\nEnter the number of the source to preprocess: ").strip()

if user_choice not in SOURCE_OPTIONS:
    raise ValueError("Invalid selection. Please rerun and choose a valid option.")

selected_source = SOURCE_OPTIONS[user_choice]

# -------------------------------------------------
# Extract selected values
# -------------------------------------------------
SOURCE_NAME = selected_source["source_name"]
DISPLAY_NAME = selected_source["display_name"]
DATASET_FOLDER = selected_source["dataset_folder"]
ZIP_FILENAME = selected_source["zip_filename"]
DATASET_CODE = selected_source["dataset_code"]
CLASS_LABEL = selected_source["class_label"]

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nSelected source configuration:")
print(f"  Display name   : {DISPLAY_NAME}")
print(f"  Dataset folder : {DATASET_FOLDER}")
print(f"  ZIP filename   : {ZIP_FILENAME}")
print(f"  Dataset code   : {DATASET_CODE}")
print(f"  Class label    : {CLASS_LABEL}")



### **Step 3 — Define ZIP, Extraction, and Output Paths**

* Builds the Google Drive path to the selected ZIP file
* Defines the local ZIP copy path in `/content`
* Defines the local extraction directory
* Defines output directories for:

  * preprocessed images
  * metadata CSV


In [ ]:
# ============================================
# Step 3: Define ZIP, Extraction, and Output Paths
# ============================================

# -------------------------------------------------
# Google Drive ZIP path
# -------------------------------------------------
zip_path = RAW_ZIP_PATHS[SOURCE_NAME]

# -------------------------------------------------
# Local extraction paths in Colab runtime
# -------------------------------------------------
local_extract_root = os.path.join(LOCAL_WORK_DIR, "raw_extracted")
local_extract_dir = os.path.join(local_extract_root, DATASET_FOLDER)

# Expected image directory after extraction
local_image_dir = os.path.join(local_extract_dir, "images")

# -------------------------------------------------
# Output paths for processed images
# -------------------------------------------------
processed_source_dir = os.path.join(LOCAL_WORK_DIR, "preprocessed", DATASET_FOLDER)
processed_image_dir = os.path.join(processed_source_dir, "images")

# -------------------------------------------------
# Output path for metadata CSV
# -------------------------------------------------
metadata_preprocessed_dir = PREPROCESSED_METADATA_DIR
metadata_csv_path = PREPROCESSED_METADATA_FILES[SOURCE_NAME]

# -------------------------------------------------
# Create required output directories
# -------------------------------------------------
os.makedirs(local_extract_root, exist_ok=True)
os.makedirs(processed_source_dir, exist_ok=True)
os.makedirs(processed_image_dir, exist_ok=True)
os.makedirs(metadata_preprocessed_dir, exist_ok=True)

# -------------------------------------------------
# Clean previous outputs for this source
# -------------------------------------------------
if os.path.exists(processed_image_dir):
    if VERBOSE:
        print(f"Clearing existing processed images: {processed_image_dir}")
    shutil.rmtree(processed_image_dir)

os.makedirs(processed_image_dir, exist_ok=True)

if os.path.exists(metadata_csv_path):
    if VERBOSE:
        print(f"Removing existing metadata file: {metadata_csv_path}")
    os.remove(metadata_csv_path)

# -------------------------------------------------
# Display resolved paths
# -------------------------------------------------
if VERBOSE:
    print("Resolved paths:")
    print(f"  ZIP path                : {zip_path}")
    print(f"  Local extract root      : {local_extract_root}")
    print(f"  Local extract dir       : {local_extract_dir}")
    print(f"  Local image dir         : {local_image_dir}")
    print(f"  Processed source dir    : {processed_source_dir}")
    print(f"  Processed image dir     : {processed_image_dir}")
    print(f"  Metadata output CSV     : {metadata_csv_path}")



### **Step Extra**


In [ ]:
# ============================================
# Step Extra: Select Source and Define Paths
# ============================================

# -------------------------------------------------
# Select source for preprocessing
# -------------------------------------------------
SOURCE_NAME = "imagenet"   # Change as needed

# -------------------------------------------------
# Resolve source-specific paths from project_config.py
# -------------------------------------------------
zip_path = RAW_ZIP_PATHS[SOURCE_NAME]

raw_extract_dir = cfg.RAW_EXTRACT_DIRS[SOURCE_NAME]
raw_image_dir = os.path.join(raw_extract_dir, "images")

processed_source_dir = PREPROCESSED_DIRS[SOURCE_NAME]
metadata_csv_path = PREPROCESSED_METADATA_FILES[SOURCE_NAME]

# -------------------------------------------------
# Ensure required output directories exist
# -------------------------------------------------
os.makedirs(processed_source_dir, exist_ok=True)
os.makedirs(PREPROCESSED_METADATA_DIR, exist_ok=True)

# -------------------------------------------------
# Optional display block
# -------------------------------------------------
if VERBOSE:
    print(f"Selected source      : {SOURCE_NAME}")
    print(f"ZIP path             : {zip_path}")
    print(f"Raw extract dir      : {raw_extract_dir}")
    print(f"Raw image dir        : {raw_image_dir}")
    print(f"Processed output dir : {processed_source_dir}")
    print(f"Metadata CSV path    : {metadata_csv_path}")



### **Step 4 — Verify, Copy, and Extract Selected ZIP File**

* Confirms that the selected ZIP file exists on Google Drive
* Copies the ZIP file locally with progress display
* Removes prior extracted content for the selected source if needed
* Extracts the ZIP locally
* Detects the actual image directory after extraction
* Verifies that image files are present

In [ ]:
# ============================================
# Step 4: Verify, Copy, and Extract Selected ZIP File
# ============================================

# -------------------------------------------------
# Verify ZIP file exists on Google Drive
# -------------------------------------------------
if not os.path.exists(zip_path):
    raise FileNotFoundError(f"ZIP file not found: {zip_path}")

zip_size_gb = os.path.getsize(zip_path) / (1024 ** 3)
print(f"ZIP file found on Google Drive: {zip_path}")
print(f"ZIP size: {zip_size_gb:.2f} GB")

# -------------------------------------------------
# Define local ZIP copy path
# -------------------------------------------------
local_zip_path = os.path.join(LOCAL_WORK_DIR, ZIP_FILENAME)

# -------------------------------------------------
# Copy ZIP from Google Drive to local runtime
# with progress display
# -------------------------------------------------
def copy_with_progress(src, dst, chunk_size=10 * 1024 * 1024):
    total_size = os.path.getsize(src)
    copied = 0
    last_reported_percent = -1

    with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
        while True:
            chunk = fsrc.read(chunk_size)
            if not chunk:
                break

            fdst.write(chunk)
            copied += len(chunk)

            percent = int((copied / total_size) * 100)
            if percent != last_reported_percent:
                print(
                    f"\rCopying ZIP: {percent:3d}% "
                    f"({copied // (1024**2)} MB / {total_size // (1024**2)} MB)",
                    end=""
                )
                last_reported_percent = percent

    print("\nCopy complete.")

if os.path.exists(local_zip_path):
    print(f"Removing existing local ZIP copy: {local_zip_path}")
    os.remove(local_zip_path)

print(f"Copying ZIP to local runtime: {local_zip_path}")
copy_with_progress(zip_path, local_zip_path)

# -------------------------------------------------
# Remove old extracted copy of this dataset, if present
# -------------------------------------------------
if os.path.exists(local_extract_dir):
    print(f"Removing existing extracted folder: {local_extract_dir}")
    shutil.rmtree(local_extract_dir)

# -------------------------------------------------
# Extract ZIP locally
# -------------------------------------------------
print(f"Extracting ZIP to: {local_extract_root}")

with zipfile.ZipFile(local_zip_path, "r") as zip_ref:
    zip_ref.extractall(local_extract_root)

print("Extraction complete.")

# -------------------------------------------------
# Inspect extracted structure
# -------------------------------------------------
print("\nTop-level extracted contents:")
top_level_items = sorted(os.listdir(local_extract_root))
for item in top_level_items[:20]:
    print(f"  {item}")
if len(top_level_items) > 20:
    print(f"  ... ({len(top_level_items) - 20} more items)")

# -------------------------------------------------
# Locate folder containing images
# -------------------------------------------------
valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

candidate_dirs = []

# 1. Expected path: /content/raw_extracted/<dataset>/images
expected_image_dir = os.path.join(local_extract_dir, "images")
if os.path.isdir(expected_image_dir):
    candidate_dirs.append(expected_image_dir)

# 2. Dataset folder itself may directly contain images
if os.path.isdir(local_extract_dir):
    candidate_dirs.append(local_extract_dir)

# 3. Search recursively for any directory containing image files
for root, dirs, files in os.walk(local_extract_root):
    image_count = sum(1 for f in files if f.lower().endswith(valid_extensions))
    if image_count > 0:
        candidate_dirs.append(root)

# Remove duplicates while preserving order
seen = set()
candidate_dirs = [d for d in candidate_dirs if not (d in seen or seen.add(d))]

if not candidate_dirs:
    raise FileNotFoundError(
        "No directory containing image files was found after extraction."
    )

# Choose the directory with the most image files
best_dir = None
best_count = 0

for d in candidate_dirs:
    try:
        count = sum(
            1 for f in os.listdir(d)
            if os.path.isfile(os.path.join(d, f)) and f.lower().endswith(valid_extensions)
        )
    except Exception:
        count = 0

    print(f"Candidate image directory: {d}  --> {count} image files")

    if count > best_count:
        best_count = count
        best_dir = d

if best_dir is None or best_count == 0:
    raise FileNotFoundError(
        "A directory was found, but no valid image files were located."
    )

local_image_dir = best_dir
extracted_files = sorted([
    f for f in os.listdir(local_image_dir)
    if os.path.isfile(os.path.join(local_image_dir, f))
    and f.lower().endswith(valid_extensions)
])

print(f"\nSelected image directory: {local_image_dir}")
print(f"Number of extracted image files found: {len(extracted_files)}")

if len(extracted_files) == 0:
    raise ValueError("No valid image files were found in the selected image directory.")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nCell 4 completed successfully.")
print(f"Local ZIP copy      : {local_zip_path}")
print(f"Extraction root     : {local_extract_root}")
print(f"Selected image dir  : {local_image_dir}")



### **Step 5 — Initialize Preprocessing Parameters**

* Loads preprocessing settings from config
* Initializes counters and metadata storage
* Assumes a fresh run for the selected source

In [ ]:
# ============================================
# Step 5: Initialize Preprocessing Parameters
# ============================================

# -------------------------------------------------
# Preprocessing settings from project_config.py
# -------------------------------------------------
target_size = TARGET_SIZE
output_format = OUTPUT_IMAGE_FORMAT
output_color_mode = OUTPUT_COLOR_MODE
output_extension = OUTPUT_FILE_EXTENSION

valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

# -------------------------------------------------
# Initialize processing counters (per run)
# -------------------------------------------------
total_files_found = len(extracted_files)
processed_count = 0
skipped_invalid_count = 0
error_count = 0

# No "skipped_existing" needed in fresh run
skipped_existing_count = 0

# -------------------------------------------------
# Initialize metadata container
# -------------------------------------------------
metadata_records = []

# -------------------------------------------------
# Initialize filename index (always start fresh)
# -------------------------------------------------
current_index = 1

# -------------------------------------------------
# Display initialization summary
# -------------------------------------------------
print("Preprocessing parameters initialized:")
print(f"  Selected source       : {DISPLAY_NAME}")
print(f"  Dataset code          : {DATASET_CODE}")
print(f"  Class label           : {CLASS_LABEL}")
print(f"  Target size           : {target_size}")
print(f"  Output format         : {output_format}")
print(f"  Output color mode     : {output_color_mode}")
print(f"  Output extension      : {output_extension}")
print(f"  Extracted files found : {total_files_found}")
print(f"  Starting index        : {current_index}")



### **Step 6 — Helper Functions**

* Defines reusable functions for:

  * validating image files
  * loading images safely
  * preprocessing images
  * saving processed images
  * building metadata records
* Preserves the existing raw stored filename for the processed output filename

In [ ]:
# ============================================
# Step 6: Helper Functions
# ============================================

# -------------------------------------------------
# Use existing filename (no renaming)
# -------------------------------------------------
def get_output_filename(input_filename):
    return input_filename

# -------------------------------------------------
# Check whether a file is a valid image candidate
# -------------------------------------------------
def is_valid_image_file(filename):
    return filename.lower().endswith(valid_extensions)

# -------------------------------------------------
# Load and validate image
# -------------------------------------------------
def load_image(image_path):
    try:
        img = Image.open(image_path)
        img.load()  # force full read to catch truncated/corrupt files
        return img
    except Exception as e:
        print(f"Skipping unreadable image: {image_path} | Error: {e}")
        return None

# -------------------------------------------------
# Preprocess image
# - resize
# - convert to grayscale
# -------------------------------------------------
def preprocess_image(img, target_size, output_color_mode):
    try:
        original_width, original_height = img.size

        if output_color_mode == "L":
            img = img.convert("L")
        else:
            img = img.convert(output_color_mode)

        img = img.resize(target_size)

        return img, original_width, original_height

    except Exception as e:
        print(f"Preprocessing failed: {e}")
        return None, None, None

# -------------------------------------------------
# Save processed image safely
# -------------------------------------------------
def save_processed_image(img, output_path, output_format):
    try:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        img.save(output_path, format=output_format)
        return True
    except Exception as e:
        print(f"Failed to save image: {output_path} | Error: {e}")
        return False

# -------------------------------------------------
# Build metadata record for one processed image
# -------------------------------------------------
def build_metadata_record(
    output_filename,
    original_width,
    original_height,
    processed_path
):
    return {
        "filename": output_filename,
        "label": CLASS_LABEL,
        "dataset_code": DATASET_CODE,
        "source_name": SOURCE_NAME,
        "source_id": "",
        "source_ref": "",
        "original_width": original_width,
        "original_height": original_height,
        "processed_path": processed_path,
    }



### **Step 7 — Preprocess Images for the Selected Source**

* Iterates through extracted raw images
* Applies preprocessing to each valid image
* Saves processed images using the same filename
* Records metadata for each processed image
* Displays progress during execution
* Reports a final preprocessing summary

In [ ]:
# ============================================
# Step 7: Preprocess Images for the Selected Source
# ============================================

# Ensure deterministic ordering
extracted_files = sorted(extracted_files)

# -------------------------------------------------
# Reset per-run counters
# -------------------------------------------------
total_files_found = len(extracted_files)
processed_count = 0
skipped_existing_count = 0
skipped_invalid_count = 0
error_count = 0

metadata_records = []

progress_interval = 100

print("Starting preprocessing...\n")

# -------------------------------------------------
# Process each extracted image
# -------------------------------------------------
for i, input_filename in enumerate(extracted_files, start=1):

    input_path = os.path.join(local_image_dir, input_filename)

    # Skip non-image files
    if not is_valid_image_file(input_filename):
        skipped_invalid_count += 1
        continue

    # Output filename = input filename
    output_filename = get_output_filename(input_filename)
    output_path = os.path.join(processed_image_dir, output_filename)

    # Reentrancy: skip if already processed
    if os.path.exists(output_path):
        skipped_existing_count += 1
        continue

    # Load image
    img = load_image(input_path)
    if img is None:
        skipped_invalid_count += 1
        continue

    # Preprocess image
    processed_img, original_width, original_height = preprocess_image(
        img,
        target_size,
        output_color_mode
    )

    if processed_img is None:
        skipped_invalid_count += 1
        continue

    # Save processed image
    save_success = save_processed_image(
        processed_img,
        output_path,
        output_format
    )

    if not save_success:
        error_count += 1
        continue

    # Build metadata record
    metadata_record = build_metadata_record(
        output_filename=output_filename,
        original_width=original_width,
        original_height=original_height,
        processed_path=output_path
    )

    metadata_records.append(metadata_record)

    processed_count += 1

    # -------------------------------------------------
    # Progress display
    # -------------------------------------------------
    if i % progress_interval == 0 or i == total_files_found:
        percent = (i / total_files_found) * 100

        print(
            f"[{i}/{total_files_found}] "
            f"{percent:6.2f}% | "
            f"Processed: {processed_count} | "
            f"Skipped(existing): {skipped_existing_count} | "
            f"Invalid: {skipped_invalid_count} | "
            f"Errors: {error_count}"
        )

# -------------------------------------------------
# Final summary
# -------------------------------------------------
print("\nPreprocessing complete.")
print(f"  Total extracted files    : {total_files_found}")
print(f"  Successfully processed   : {processed_count}")
print(f"  Skipped existing         : {skipped_existing_count}")
print(f"  Skipped invalid/corrupt  : {skipped_invalid_count}")
print(f"  Save/processing errors   : {error_count}")
print(f"  Metadata records created : {len(metadata_records)}")



### **Step 8 — Save Preprocessed Metadata CSV**

* Converts metadata records into a DataFrame
* Applies the expected metadata column structure
* Saves the per-source metadata CSV

In [ ]:
# ============================================
# Step 8: Save Preprocessed Metadata CSV
# ============================================

import pandas as pd

# -------------------------------------------------
# Verify metadata exists
# -------------------------------------------------
if len(metadata_records) == 0:
    raise ValueError("No metadata records to save. Check preprocessing step.")

# -------------------------------------------------
# Create DataFrame
# -------------------------------------------------
df_metadata = pd.DataFrame(metadata_records)

# -------------------------------------------------
# Enforce column order from config
# -------------------------------------------------
expected_columns = PREPROCESSED_METADATA_COLUMNS

missing_columns = [col for col in expected_columns if col not in df_metadata.columns]

if missing_columns:
    raise ValueError(f"Missing expected metadata columns: {missing_columns}")

df_metadata = df_metadata[expected_columns]

# -------------------------------------------------
# Save CSV
# -------------------------------------------------
os.makedirs(os.path.dirname(metadata_csv_path), exist_ok=True)

df_metadata.to_csv(metadata_csv_path, index=False)

# -------------------------------------------------
# Display summary
# -------------------------------------------------
print("Metadata CSV saved successfully.")
print(f"  Output file        : {metadata_csv_path}")
print(f"  Number of records  : {len(df_metadata)}")
print(f"  Columns            : {list(df_metadata.columns)}")



### **Step 9 — Processing Summary and Validation**

* Reports processing totals
* Confirms output image counts
* Confirms metadata record counts
* Verifies consistency between processed images and metadata

In [ ]:
# ============================================
# Step 9: Processing Summary and Validation
# ============================================

# -------------------------------------------------
# Basic output checks
# -------------------------------------------------
processed_output_files = sorted([
    f for f in os.listdir(processed_image_dir)
    if os.path.isfile(os.path.join(processed_image_dir, f))
    and f.lower().endswith(output_extension)
])

metadata_record_count = len(df_metadata)

# -------------------------------------------------
# Display run summary
# -------------------------------------------------
print("Processing summary:")
print(f"  Selected source         : {DISPLAY_NAME}")
print(f"  Dataset code            : {DATASET_CODE}")
print(f"  Class label             : {CLASS_LABEL}")
print(f"  Extracted input files   : {total_files_found}")
print(f"  Newly processed images  : {processed_count}")
print(f"  Invalid/corrupt skipped : {skipped_invalid_count}")
print(f"  Errors                  : {error_count}")
print(f"  Output image files      : {len(processed_output_files)}")
print(f"  Metadata records        : {metadata_record_count}")
print(f"  Processed image dir     : {processed_image_dir}")
print(f"  Metadata CSV path       : {metadata_csv_path}")

# -------------------------------------------------
# Validation checks
# -------------------------------------------------
if processed_count != len(processed_output_files):
    print("\nWARNING: Processed image count does not match output file count.")

if processed_count != metadata_record_count:
    print("\nWARNING: Processed image count does not match metadata record count.")

if len(processed_output_files) != metadata_record_count:
    print("\nWARNING: Output file count does not match metadata record count.")

if (
    processed_count == len(processed_output_files)
    and processed_count == metadata_record_count
):
    print("\nValidation passed: image count and metadata count are consistent.")

# -------------------------------------------------
# Preview first few output filenames
# -------------------------------------------------
preview_count = min(5, len(processed_output_files))

if preview_count > 0:
    print(f"\nFirst {preview_count} processed output files:")
    for fname in processed_output_files[:preview_count]:
        print(f"  {fname}")

# -------------------------------------------------
# Confirm metadata file exists
# -------------------------------------------------
if os.path.exists(metadata_csv_path):
    print("\nMetadata CSV verified.")
else:
    print("\nWARNING: Metadata CSV file was not found.")



### **Step 10 — Raw vs. Preprocessed Comparison**

* Displays **3 sample pairs**
* Shows the raw image on the left
* Shows the corresponding preprocessed image on the right
* Confirms resize, grayscale conversion, and filename consistency

In [ ]:
# ============================================
# Step 10: Raw vs Preprocessed Comparison
# ============================================

import matplotlib.pyplot as plt

# -------------------------------------------------
# Select a few processed images for comparison
# -------------------------------------------------
preview_files = sorted([
    f for f in os.listdir(processed_image_dir)
    if os.path.isfile(os.path.join(processed_image_dir, f))
    and f.lower().endswith(output_extension)
])

num_preview = min(3, len(preview_files))   # <-- limit to 3

if num_preview == 0:
    print("No processed images available for preview.")
else:
    print(f"Displaying {num_preview} raw vs preprocessed image comparisons...\n")

    plt.figure(figsize=(12, 4 * num_preview))

    for i, filename in enumerate(preview_files[:num_preview], start=1):

        raw_path = os.path.join(local_image_dir, filename)
        processed_path = os.path.join(processed_image_dir, filename)

        # Load raw image
        try:
            with Image.open(raw_path) as raw_img:
                raw_img_display = raw_img.copy()
                raw_size = raw_img.size
        except Exception:
            raw_img_display = None
            raw_size = "Unavailable"

        # Load processed image
        try:
            with Image.open(processed_path) as proc_img:
                proc_img_display = proc_img.copy()
                proc_size = proc_img.size
        except Exception:
            proc_img_display = None
            proc_size = "Unavailable"

        # -------------------------------------------------
        # Raw image (left)
        # -------------------------------------------------
        plt.subplot(num_preview, 2, 2 * i - 1)

        if raw_img_display is not None:
            plt.imshow(raw_img_display)
            plt.title(f"Raw: {filename}\nSize: {raw_size}")
        else:
            plt.text(0.5, 0.5, "Raw image load failed", ha="center", va="center")

        plt.axis("off")

        # -------------------------------------------------
        # Preprocessed image (right)
        # -------------------------------------------------
        plt.subplot(num_preview, 2, 2 * i)

        if proc_img_display is not None:
            plt.imshow(proc_img_display, cmap="gray")
            plt.title(f"Preprocessed: {filename}\nSize: {proc_size}")
        else:
            plt.text(0.5, 0.5, "Preprocessed image load failed", ha="center", va="center")

        plt.axis("off")

    plt.tight_layout()
    plt.show()

